In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib

params = urllib.parse.quote_plus(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=localhost\\SQLEXPRESS;'
    'DATABASE=course_project_1;'
    'Trusted_Connection=yes;'
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")



In [2]:
customer = pd.read_sql("SELECT CustomerID, Customer_Name FROM Customers",engine)

sales = pd.read_sql(
    "SELECT SaleID, CustomerID, BranchID, SaleDate, TotalAmount FROM Sales", engine, parse_dates=["SaleDate"]
)
branches = pd.read_sql("SELECT BranchID, Branch_Name FROM Branches",engine)

saleDetails = pd.read_sql("SELECT SaleID, ProductID, Quantity FROM SaleDetails",engine)

products = pd.read_sql("SELECT ProductID, Product_Name, Price FROM Products",engine)

sales_branch = sales.merge(branches[["BranchID", "Branch_Name"]], on = "BranchID", how = "left")
products_sd = (saleDetails.merge
    (products[["ProductID", "Product_Name", "Price"]],on="ProductID", how="left"))

M:\AMIT_AI_Diploma\AMIT_AI_Diploimint\env\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


In [1]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017")
mongo_db = client["Data_analysis_project"]
sales_collection = mongo_db["Sales_Collection"]
sales_collection.drop()

In [6]:
for _ ,cust in customer.iterrows():
    cid = cust["CustomerID"]
    cname = cust["Customer_Name"]

    cust_sales = sales_branch[sales_branch["CustomerID"] == cid]
    if cust_sales.empty:
        doc = {
            "CustomerID": cid,
            "CustomerName": cname,
            "TotalSpent": 0.0,
            "TotalOrders": 0,
            "TopProducts": [],
            "PreferredBranch": None,
            "MonthlyPurchases": {}
        }
        sales_collection.insert_one(doc)
        continue
    total_spent = float(cust_sales["TotalAmount"].sum())
    total_orders = int(cust_sales["SaleID"].nunique())

    # top products

    sales_sd = cust_sales["SaleID"].unique()
    cust_sd = products_sd[products_sd["SaleID"].isin(sales_sd)]
    top_products_df = (cust_sd.groupby(
        ["Product_Name", "Price"])["Quantity"].sum().reset_index().sort_values("Quantity",ascending=False).head(5))
    top_products = [
        {"Product_Name":row["Product_Name"], "Price":int(row["Price"]), "Quantity":int(row["Quantity"])}
        for _, row in top_products_df.iterrows()
    ]

    # PreferredBranch

    PreferredBranch_df =(
        cust_sales.groupby(["BranchID", "Branch_Name"])["SaleID"].nunique().reset_index(name="Orders").sort_values("Orders", ascending=False)
    )
    preferred_branch = PreferredBranch_df.iloc[0]["Branch_Name"]

    #

    tmp = cust_sales.copy()
    tmp["YearMounth"] = tmp["SaleDate"].dt.to_period("M").astype(str)
    monthly = (tmp.groupby("YearMounth")["TotalAmount"].sum().to_dict())
    monthly ={k:float(v) for k, v in monthly.items()}


    #

    document = {
        "CustomerID": cid,
        "CustomerName": cname,
        "TotalSpent": total_spent,
        "TotalOrders": total_orders,
        "TopProducts": top_products,
        "PreferredBranch": preferred_branch,
        "MonthlyPurchases": monthly
    }

    sales_collection.insert_one(document)

print("SalesCollection created and filled.")

SalesCollection created and filled.
